# CS336 Assignment 1

## 实现BPE

### UNICODE

在UNICODE标准中，每个字符对应一个代码

In [3]:
ord("牛")
chr(29275)

'牛'

chr(0) 代表什么

In [ ]:
chr(0)
print(chr(0))

 


In [6]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [7]:
print("this is a test" + chr(0) + "string")

this is a test string


UNICODE标准中的0 代表空字符(NUL)，在python中， print会输出它，但通常终端不会显示出来
- 在C语言中用该字符作为字符串的结束标记

#### UNICODE 编码

将UNICODE字符用字节序列来进行编码，以解决词汇表太大的问题
- 直接使用UNICODE标准的代码，这个数字范围很大，几个UNICODE字符对应多少个代码

In [2]:
test_string0 = "A"
utf8_encoded = test_string0.encode("utf-16")
print(utf8_encoded)

test_string1 = "你好"
utf8_encoded = test_string1.encode("utf-8")
print(utf8_encoded)

b'\xff\xfeA\x00'
b'\xe4\xbd\xa0\xe5\xa5\xbd'


在解码时，UTF-8按照首字节的前缀，决定每个字符长度
- 0: 表示单字节字符，对应ASCII 字符
- 110: 后面跟一个续字节
- 1110: 后面跟两个
- 11110: 后面跟三个
- 续字节统一是10

因此一个UNICODE字符，并不对应一个字节
- 只有ASCII 字符是一个字符对应一个字节

而且这样所有的UNICODE字符我们都能用 0-255 来表示了，只要它们遵循统一的编码方式，如UTF-8
- 词汇表大小只有256

为什么 tokenizer 用UTF-8编码而不是UTF-16或UTF-32
- 因为ASCII在UTF-8 都是1字节，对英文场景天然更节省长度
- UTF-16 的基础单位是16位，但实际上一个可见字符可能占1个或2个单位

In [13]:
test_string= "hello 世界"
print(test_string.encode("utf-8"))
print(len(test_string.encode("utf-8")))
print(test_string.encode("utf-16"))
print(len(test_string.encode("utf-16")))


b'hello \xe4\xb8\x96\xe7\x95\x8c'
12
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00 \x00\x16NLu'
18


In [3]:
# A wrong func

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8")) ## no wrong
# decode_utf8_bytes_to_str_wrong("你好".encode("utf-8")) ## wrong


'hello'

这个函数错误的原因也就是前面提到的UTF-8编码的原理，除了ASCII，其他字符都不是一字节的

### 基于subword 的tokenization

由于基于字节的tokenization，会导致序列长度过长，也不是很适合

一个自然的思路就是，基于词出现的频率，将经常出现的，合并起来
- BPE

### BPE Tokenizer 训练

一般包括三步
1. 词表初始化: 词表是 token 到 整数的一一映射, 对于我们的字节级BPE分词器, 初始词表大小256, 对应一字节的所有可能取值
2. pre-tokenization

In [3]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

import regex as re

re.findall(PAT, "low lower")

['low', ' lower']

train_bpe
1. 读文件
   1. 输入：文件路径
   2. 输出：文本对象
2. 剥掉special token
   1. 输入：整个文本
   2. 输出：按special token 切成文档边界的预料
3. 正则预分词
   1. 输入：文本
   2. 输出：list
4. 每个词转UTF-8 字节
   1. 输入：str
   2. 输出：utf-8 表示
5. 统计相邻字节对频率
   1. 输入：整份语料
   2. 输出：计数表
6. 合并循环
   1. 输入：计数表 + 词表
   2. 输出：查计数表 -> 找频率最高的pair，把这对计入合并历史，在语料里把所以出现这个相邻对的地方替换成新词条，词表+1 检查是否达到vocab_size
7. 组装输出
   1. 输入：合并历史
   2. 输出：合并循环记下的所有oair，按创建顺序。256个单字节 + 新词表 + special token

In [8]:
old_tuple = (b'a', b'a', b'c')

pair = (b'a', b'a')

i = 0

new_tuple = []
while i < len(old_tuple):
    if i + 1 < len(old_tuple) and (old_tuple[i], old_tuple[i + 1]) == pair:
        new_tuple.append(old_tuple[i] + old_tuple[i+1])
        i += 2
    else:
        new_tuple.append(old_tuple[i])
        i += 1

new_tuple = tuple(new_tuple)
print(new_tuple)

(b'aa', b'c')


```
start = time.perf_counter()
vocab, merges = train_bpe('data/TinyStoriesV2-GPT4-train.txt', 10000, ['<|endoftext|>'])
end = time.perf_counter()
```

```
❯ uv run cs336_basics/tokenizer.py
72.07682266703341
```